In [ ]:
cd ..

In [ ]:
from src.utils import log, CustomException
import pandas as pd
import json
log = log()

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('BAAI/bge-base-en-v1.5')
tokenizer = model.tokenizer

In [ ]:
import sys
import json
try:
    log.info("Loading cleaned EU MDR 2017-745 documents for token length analysis.")
    with open('data/processed/cleaned_eu_mdr_2017-745.json', 'r') as f:
        docs = json.load(f)
    
    structure_prefix = ""
    part_pattern = "\nPART [A-Z] \n"
    simple_pattern = r'\n(\d+)\.\s*\n'
    decimal_pattern = "\n\d+\.\d+\.\s*\n"
    triple = "\n\d+\.\d+\.\d+\.\s*\n"

    for i, doc in enumerate(docs):
        page_content = doc.get('page_content')
        metadata = doc.get('metadata')
        token_length = len(tokenizer.encode(page_content))
      

            
except Exception as e:
    log.exception(f"An error occurred: {e}")
    raise CustomException(e, sys)

In [ ]:
for doc in docs:
    doc.get('page_content', '')

In [ ]:
info = docs[148].get('page_content', '')

In [ ]:
print(info)

In [ ]:
import re
pattern = r'\n(\d+)\.\s*\n(.+)'
part_pattern = "\nPART [A-Z] \n(.+)"
matches =  re.finditer(part_pattern, docs[147].get('page_content'))

In [ ]:
len(tokenizer.encode(docs[147].get('page_content', '')))

In [ ]:
patterns = {
    "part"        : r'\nPART [A-Z] \n(.+)',
    "simple"      : r'\n(\d+)\.\s*\n(.+)',
    "decimal"     : r'(?:^|\n)(\d+\.\d+\.)(?!\d)\s+(.+)',   
    "triple"      : r'(?:^|\n)(\d+\.\d+\.\d+\.)\s+(.+)',
    "paren_letter": r'(?:^|\n)\(([a-z])\)\s+(.+)',          
    "paren_num"   : r'(?:^|\n)\((\d+)\)\s+(.+)',  
    "bullet"      : r'(?:^|\n)(—)\s+(.+)',          
}

marker_patterns = {"pattern_annex" : "^(ANNEX [IVX]+) \n(.+)", "pattern_chapter" : "^(CHAPTER [IVX]+) \n(.+)", "pattern_section" : "^(SECTION [0-9]+) \n(.+)", "pattern_article" :"^(Article [0-9]+) \n(?!Article)(?!— )(.+)"}


In [ ]:
def token_length(page_content):
    return len(tokenizer.encode(page_content))
def get_split_levels(page_content):
    levels = []
    if token_length(page_content) <= 512:
        return ['no_split']
    if re.search(patterns['part'], page_content):
        levels.append("part")
    if re.search(patterns['simple'], page_content):
        levels.append("simple")
    if re.search(patterns['decimal'], page_content):
        levels.append("decimal")
    if re.search(patterns['triple'], page_content):
        levels.append("triple")
    if re.search(patterns['paren_letter'], page_content):
        levels.append("paren_letter")
    if re.search(patterns['paren_num'], page_content):
        levels.append("paren_num")
    if re.search(patterns['bullet'], page_content):
        levels.append("bullet")
    return levels

In [ ]:
def get_text_piece(pattern, text):
    find = re.finditer(pattern, text, re.M)
    matches = []
    pieces = []
    for match in find:
        matches.append((match.start(), match.group()))
    if not matches:
        return [text]
    if matches and matches[0][0] > 0:
        pieces.insert(0, text[0:matches[0][0]])
    for i, mark in enumerate(matches):
        pattern_length = len(mark[1])
        if i < len(matches)-1:
            pieces.append(text[mark[0]:matches[i+1][0]])
        else:
            pieces.append(text[mark[0]:])
        
    return pieces

a = get_text_piece(patterns['part'], info)
lengths = [token_length(piece) for piece in a]
a


# New Approch:

Here is the summary what i have, what i want and what is current approach doing:

I already have separate section-wise chunks from pdf_extractor.

I have to take one chunk that is already splitted section or article-wise.

Process the chunk and split those chunk into 512 sub-chunks without breaking from mid-sentence.

Get the new chunk with new structural boundry follow the same process.

What current approach does:

It splits eacxh chunk by levels not by token length.

So, when level changes it creates new chunk instaed of appending it until 512 or less.

it should start at new level only if it's exceeds then 512 or 500

In [ ]:
def create_chunks(text, metadata):
    return {'page_content': text, 'metadata': metadata}

In [ ]:
def pack(chunks, max_tokens=512):
    packed = []
    buffer_text = ""
    buffer_meta=dict()
    buffer_pages = []

    def flush():
        buffer_meta['token_length'] = token_length(buffer_text)
        packed.append(create_chunks(buffer_text, buffer_meta))

    for i, c in enumerate(chunks):
        candidate = buffer_text + "\n" + c['page_content'] if buffer_text else c['page_content']
        page = c['metadata'].get('page_number')

        if token_length(candidate) > max_tokens and buffer_text:

            flush()
            buffer_text = c['page_content']
            buffer_meta = c['metadata'].copy()
            
        else:
            if not buffer_text:
                buffer_meta= c['metadata'].copy()
            buffer_text = candidate
       

    if buffer_text:
        flush()
    total = len(packed)
    for position, chunk in enumerate(packed, start=1):
        chunk['metadata']['subchunk_id'] = position
        chunk['metadata']['total_subchunks'] = total
    
    return packed

            

In [ ]:
def split_document(text, metadata, levels, prefix=""):
    if token_length(text) <= 512:
        if prefix:
            final_text = prefix + "\n" + text
        else:
            final_text = text
        
        new_metadata = metadata.copy()
        new_metadata['token_length'] = token_length(final_text)
        return [create_chunks(final_text, new_metadata)]
    
    if not levels:
        if prefix:
            final_text = prefix + "\n" + text
        else:
            final_text = text
        new_metadata = metadata.copy()
        new_metadata['token_length'] = token_length(final_text)
        return [create_chunks(final_text, new_metadata)]
    
    current_level = levels[0]
    remaining_levels = levels[1:]

    text_pieces = get_text_piece(patterns[current_level], text)
    if not text_pieces:
        text_pieces = [text]
    sub_chunks = []
    for i, piece in enumerate(text_pieces):
        '''if any(re.match(pattern, piece) for pattern in marker_patterns.values()):
            continue'''

        match = re.match(patterns[current_level],piece)
        
        if match:
            header = match.group()
            new_prefix = prefix + "-" + header if prefix else header
            piece = piece.replace(header, "")
        
        else:
            new_prefix = prefix

        sub_chunks.extend(split_document(piece, metadata, remaining_levels, prefix=new_prefix))
    return pack(sub_chunks, max_tokens=512)

In [ ]:
all_chunks = []
for i, doc in enumerate(docs):
    page_content = doc.get('page_content', '')
    metadata = doc.get('metadata', {})
    levels = get_split_levels(page_content)
    chunks = split_document(page_content, metadata, levels)
    all_chunks.extend(chunks)

print(f"Total chunks: {len(all_chunks)}")
over_512 = [c for c in all_chunks if c['metadata']['token_length'] > 512]
print(f"Chunks over 512: {len(over_512)}")

In [ ]:
all_chunks

In [ ]:
over_512 = [c for c in all_chunks if c['metadata']['token_length'] > 512]
for c in over_512:
    print(c['metadata']['token_length'], 
          c['metadata'].get('annex',''), 
          c['metadata'].get('article',''),
          c['metadata'].get('page_num'),
          c['page_content'][:500])
    print()

In [ ]:
import re

def _norm(s):
    return " ".join(str(s or "").split())   # collapses \n and repeated spaces

def is_header_only_doc(doc):
    body = _norm(doc['page_content'])
    m = doc['metadata']

    def field(key):
        return _norm(m.get(key, ''))

    # metadata match — build candidates with AND without a space, both normalized
    candidates = []
    for mk, tk in (('section', 'section_title'),
                   ('chapter', 'chapter_title'),
                   ('annex',   'annex_title')):
        marker, title = field(mk), field(tk)
        if marker or title:
            candidates.append(_norm(marker + " " + title))
            candidates.append(_norm(marker + title))
    title_only = body in candidates

    bare = re.fullmatch(
        r'(?:(?:Article\s+\d+|ANNEX\s+[IVXLC]+|CHAPTER\s+[IVXLC]+|SECTION\s+\d+)\s+)*'  # leading noise
        r'(?:SECTION\s+\d+|CHAPTER\s+[IVXLC]+|ANNEX\s+[IVXLC]+)\s+'                     # real marker
        r'[A-Z][A-Za-z0-9 ,()&/\'’\-–]+',                                              # title, no periods
        body
    )
    return bool(title_only or bare)

docs = [d for d in all_chunks if not is_header_only_doc(d)]

In [ ]:
print(docs)

In [ ]:
p = []

for i, c in enumerate(docs):
    d = {
        "Chunk_no": i,
        "token_length": c["metadata"]["token_length"]
    }
    p.append(d)

In [ ]:
data = pd.DataFrame(p)
import numpy as np

bins = np.arange(0, 550, 20)

data['token_length'].plot(kind='hist', bins=bins, edgecolor='black',xlabel="token_length")



In [ ]:
over_512 = [c for c in all_chunks if c['metadata']['token_length'] < 40 ]
for c in over_512:
    print( 
          c['metadata'].get('annex',''), 
          c['metadata'].get('article',''),
          c['page_content'][:500],
          )
    print()

In [ ]:
with open('data/processed/chunks/chunks.json', 'w') as f:
    json.dump(docs, f, indent=4)

In [ ]:
cd ..

In [ ]:
import json

with open('data/processed/chunks/chunks.json', 'r') as f:
    docs = json.load(f)

In [ ]:
print(docs)

# Build the Embedding setup

In [ ]:
import chromadb

## Creating the persistent database

In [ ]:
client  = chromadb.PersistentClient(path = "./data/database/chromadb/")

In [ ]:
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
import torch
try:
    device = "cuda" if torch.cuda.is_available() else 'cpu'
    embedding_fn = SentenceTransformerEmbeddingFunction(model_name='BAAI/bge-base-en-v1.5', device=device)
except Exception as e:
    log.exception(f"An Error occured: {e}")
    raise CustomException(e, sys)

In [ ]:
#client.delete_collection(name="mdr_1")
collection = client.get_or_create_collection(name = "eu_mdr", 
                                     embedding_function=embedding_fn,
                                     configuration={
                                                "hnsw": {
                                                    "space": "cosine",
                                                    "ef_construction": 300
                                                }
                                            })

In [ ]:
collection.configuration

### Evaluating Small

* Add few chunks to collection
* Evaluate with a query and check the semantic match
* check what it returns: Distance or similarity?

In [ ]:
import hashlib
ids, pcs, metadatas = [],[],[]
for doc in docs:
    metadata = doc.get('metadata')
    metadatas.append(metadata)
    pc = doc.get('page_content')
    pcs.append(pc)
    chunk_id = hashlib.sha256(pc.encode('utf-8')).hexdigest()
    ids.append(chunk_id)

collection.add(ids=ids, 
                   documents=pcs,
                   metadatas=metadatas)


In [ ]:
import hashlib
ids1, pcs, metadatas = [],[],[]
for doc in docs:
    metadata = doc.get('metadata')
    metadatas.append(metadata)
    pc = doc.get('page_content')
    pcs.append(pc)
    chunk_id = hashlib.sha256(pc.encode('utf-8')).hexdigest()
    ids1.append(chunk_id)


    

In [ ]:
print(len(ids), len(ids1))
print("identical:", ids == ids1)

In [ ]:
len(ids) == len(set(ids))

### Confirm the colletion count

In [ ]:
collection.count()

### Confirm the embedding dimensions

In [ ]:
# Pullng one embedding
collection.get(include=['embeddings'], ids=["2fa14721893766c74fe238401df190da4d802fd600f9fccdcda37781ce6143e4"]).get('embeddings').shape

### Querying from the added chunks to get the self-match distance

In [ ]:
query_1 = collection.query(
    query_texts=["Why did the EU replace the old medical device directives?"],
    n_results=3
)

In [ ]:
query_1.get('metadatas')

In [ ]:
query_1.get('distances')

### Querying with negative query(unrelated question)

In [ ]:
query_2 = collection.query(
    query_texts=["What are the UDI carrier placement rules for reusable devices?"],
    n_results=3
)

In [ ]:
query_2.get('metadatas')

In [ ]:
query_2.get('distances')

## Observations:
1. The collection is persistent and store the chunks efficiently.
2. The Embedding function works well and generates expected embeddings dimensions which is 768.
3. The chromadb's retrieval fetches the semantically matched information and gices efficient distance scores.


## Scalling to all the chunks
* Delete and rebuild the collections

* Build the lists across all the chunks
 
* Assert Uniqueness

* Add all chunks to collection

* Confirm the collection count

### Delete-and-rebuild collection

In [ ]:
import chromadb
client  = chromadb.PersistentClient(path = "./data/database/chromadb/")

In [ ]:
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
import torch
try:
    device = "cuda" if torch.cuda.is_available() else 'cpu'
    embedding_fn = SentenceTransformerEmbeddingFunction(model_name='BAAI/bge-base-en-v1.5', device=device)
except Exception as e:
    log.exception(f"An Error occured: {e}")
    raise CustomException(e, sys)

In [ ]:
#client.delete_collection(name="mdr_1")
collection = client.get_or_create_collection(name = "eu_mdr", 
                                     embedding_function=embedding_fn,
                                     configuration={
                                                "hnsw": {
                                                    "space": "cosine",
                                                    "ef_construction": 300
                                                }
                                            })

### Build the lists across all the chunks

In [ ]:
len(docs)

In [ ]:
import hashlib

try:
    ids, metadatas, pcs= [], [], []
    for doc in docs:
        pc = doc.get('page_content')
        pcs.append(pc)
        metadatas.append(doc.get('metadata'))
        ids.append(hashlib.sha256(pc.encode("utf-8")).hexdigest())

except Exception as e:
    log.exception(f"The Error occured: {e, sys}")
    raise CustomException(e,sys)

In [ ]:
# Confirm the total docs
len(ids)

### Assert Uniqueness

In [ ]:
len(ids) == len(set(ids))

### Add to collection

In [ ]:
if len(ids) == len(set(ids)):
    collection.add(ids=ids, 
                   documents=pcs,
                   metadatas=metadatas)

### Assert the count

In [ ]:
collection.count()

### Questions:

1. Under what conditions are medical devices manufactured and used within health institutions exempt from the requirements of the regulation? Gold chunk that contains article 5
2. Compare the product categories in Annex XVI that involve modifications to the human body. Which categories involve surgical invasion, injections, electromagnetic radiation, and brain stimulation, and what are their respective purposes?
3. What shall be included by the PMCF plan?
4. What are the requirements mentioned in Article 62 for conformity of devices?
5. How can a manufacturer obtain proof that a CE-marked device can be marketed in the European Union for export purposes?(similar to previous question)
6. what is 'notified body'?
7. What are the differences in conformity assessment between Class IIa and Class IIb devices?
8. What is the definition of a medical device under MDR?
9. Where does the 'Regulation (EU) 2017/745' not apply?
10. which documents are required if one wants to perform clinincal invastigation?

In [ ]:
query_3 = collection.query(
    query_texts=["which documents are required if one wants to perform clinincal invastigation?"],
    n_results=5
)

In [ ]:
query_3

In [ ]:
eval_questions = ["Under what conditions are medical devices manufactured and used within health institutions exempt from the requirements of the regulation? Gold chunk that contains article 5",
"Compare the product categories in Annex XVI that involve modifications to the human body. Which categories involve surgical invasion, injections, electromagnetic radiation, and brain stimulation, and what are their respective purposes?",
"What shall be included by the PMCF plan?",
"What are the requirements mentioned in Article 62 for conformity of devices?",
"How can a manufacturer obtain proof that a CE-marked device can be marketed in the European Union for export purposes?",
"what is 'notified body'?",
"What are the differences in conformity assessment between Class IIa and Class IIb devices?",
"What is the definition of a medical device under MDR?",
"Where does the 'Regulation (EU) 2017/745' not apply?",
"which documents are required if one wants to perform clinincal invastigation?"]

container = []

for question in eval_questions:
    
    query = collection.query(
    query_texts=[question],
    n_results=5
    )
    
    container.append({"question": question,
        "retrieved_data": query}
                )

with open("data/evaluation/test_questions.json", 'w') as f:
    json.dump(container, f, indent=4)

In [ ]:
container

## Building Evaluation

In [ ]:
def recall_at_k(retrieved_ids, gold_chunk_ids, k):
    rid_set = set(retrieved_ids[:k])
    gid_set = set(gold_chunk_ids)
    if not len(gid_set & rid_set) == 0:
        return 1
    else: return 0

In [ ]:
def RR(retrieved_ids, gold_chunk_ids):
    rank=None
    for i, rid in enumerate(retrieved_ids):
        if rid in gold_chunk_ids:
            rank = i+1
            break
    if rank is not None:    
        return 1 / rank

    return 0

In [ ]:
from pathlib import Path
evalution_set = Path("data/evaluation/test_questions.json")


with open(evalution_set, 'r') as f:
    eval_data = json.load(f)

hits = 0
reciprocal_ranks = []
for questions in eval_data['questions']:
    question = questions['question']
    results = collection.query(query_texts=[question], n_results=5)
    gold_chunk_id = questions['gold_chunk_ids']
    retrieved_ids = results['ids'][0]
    recall = recall_at_k(retrieved_ids, gold_chunk_id, k=1)
    if recall:
        hits += 1
    reciprocal_ranks.append(RR(retrieved_ids, gold_chunk_id))

   

recall_at_1  = hits/len(eval_data['questions'])
MRR = sum(reciprocal_ranks)/len(eval_data['questions'])


In [ ]:
recall_at_1, MRR

In [ ]:
def label(number, title):
    if title:
        return f"{number} ({title})"
    return number

In [ ]:
def create_source(metadata: dict):
    article= metadata.get('article', "").strip()
    annex = metadata.get("annex","").strip()
    chapter = str(metadata.get('chapter',"")).strip()
    chapter_title = metadata.get("chapter_title", "").strip()
    annex_title = metadata.get("annex_title","").strip()
    article_title = metadata.get("article_title", "").strip()
    section = metadata.get("section","").strip()
    section_title = metadata.get("section_title").strip()
    page_range = "page no: " + str(metadata.get("page_number"))

    if chapter == "0" and chapter_title == "preamble":
        source = "EU MDR Preamble"
    
    elif article:
        parts = []
        if chapter: parts.append(label(chapter, chapter_title))
        if section: parts.append(label(section,section_title))
        if page_range: parts.append(page_range)
        parts.append(label(article, article_title))
        source =  ", ".join(parts)

    elif annex:
        parts = [label(annex, annex_title)]
        if page_range: parts.append(page_range)
        if chapter: parts.append(label(chapter, chapter_title))
        source =  ", ".join(parts)

    return source



In [ ]:
for x in container:
    f = x.get("retrieved_data")
    o = f['metadatas'][0][0]
    print(create_source(o))

In [ ]:
query_3

In [ ]:
query_3['documents'][0]

In [ ]:
# Building a function

def create_model_context(retrieved_chunks: dict) -> str:
    list_of_chunks = []
    docs = retrieved_chunks['documents'][0]
    metadatas = retrieved_chunks['metadatas'][0]
    for content, meta in zip(docs, metadatas):
        source = create_source(meta)
        page_c = "<CHUNK_SOURCE: " + source + ">\n" + content + "</CHUNK>"
        list_of_chunks.append(page_c)
    
    context = "\n-------------------------------------------------------------------------------------------------\n".join(list_of_chunks)
    return context


create_model_context(query_3)

In [ ]:
test_generator = []
for q in container:
    question = q['question']
    result = collection.query(
        query_texts=[question]
    )
    test_generator.append(
        {'question': question,
         'context': create_model_context(result)}
    )


test_generator

In [ ]:
with open("prompt.md", 'r') as f:
    prompt = f.readlines()

prompt.insert(10, 'hi')
prompt

### Building a function to merge prompt and retrieved chunks

In [ ]:
def create_prompt(md_file, question):
    results = collection.query(
        query_texts=[question]
    )
    model_context = create_model_context(results)
    with open(md_file, 'r') as f:
        prompt = f.readlines()
    index = prompt.index("<SOURCES>\n")
    prompt.insert(index+1, model_context)
    prompt.insert(len(prompt), f"<QUESTION:>\n{question} \n</QUESTION>")

    with open(md_file,'w') as c:
        c.writelines("".join(prompt))


In [ ]:
create_prompt("prompt.md", "what is Medical Device?")

In [ ]:
cd ..

In [ ]:
import json
with open("data/evaluation/question_6.json", 'r') as f:
    data = json.load(f)
for item in data.get(list(data.keys())[0])['llama3.2:3b']:
    print(f"Run: {item['run'] +1} \n" + "*"*120 + f"\n{item['answer']}\n" + "="*120)